# Tugas Praktikum — Latihan Mandiri Pertemuan 9: Classical Problems

| | |
|---|---|
| **Nama** | Diva Hersa |
| **NIM** | 2411513034 |
| **Mata Kuliah** | Pemrosesan Paralel |
| **Dosen** | Desta Yolanda, M.T |
| **Topik** | Dining Philosophers, Readers-Writers, Sleeping Barber, Refleksi |

Notebook ini berisi jawaban **4 soal Latihan Mandiri (Bagian 6)**. Setiap soal memuat: penjelasan rancangan, kode, output, dan analisis berdasarkan materi Pertemuan 9 (Coffman, starvation, bounded waiting, bounded buffer, dan rejection).

> Catatan: angka waktu/throughput dapat sedikit berbeda setiap kali sel dijalankan, karena bergantung pada penjadwalan thread oleh sistem operasi.


---
## Soal 1 — Dining Philosophers dengan N = 3, 7, dan 10

**Instruksi:** ulangi solusi *arbitrator* (Bagian 1) untuk N = 3, 7, 10; ukur throughput (total makan per detik); apakah throughput naik linear terhadap N?

**Rancangan:**
- Solusi arbitrator sama dengan materi: `Semaphore(N-1)` membatasi maksimal N-1 filsuf yang boleh berebut garpu, sehingga *circular wait* penuh tidak mungkin terjadi (memutus salah satu syarat Coffman).
- Kode materi saya bungkus menjadi fungsi `jalankan_arbitrator(N)` agar bisa dipakai untuk beberapa nilai N. Setiap filsuf makan **20 putaran**, lama makan 0,02 detik, lama berpikir 0,01 detik.
- **Throughput = total makan / durasi total.** Tiap N diulang 3 kali lalu diambil rata-ratanya.
- Saya menambahkan penghitung **jumlah filsuf yang makan bersamaan** untuk membuktikan penyebab hasilnya.
- `time.sleep()` melepaskan GIL, jadi waktu makan benar-benar bisa tumpang tindih antar-thread dan GIL bukan penghambat pada simulasi ini.


In [1]:
import threading
import time
import statistics

def jalankan_arbitrator(N, putaran=20, waktu_makan=0.02, waktu_pikir=0.01):
    """Solusi arbitrator (Semaphore N-1) untuk N filsuf. Mengembalikan (total_makan, durasi, maks_makan_bersamaan)."""
    forks = [threading.Lock() for _ in range(N)]
    kursi_meja = threading.Semaphore(N - 1)
    selesai_makan = [0] * N
    sedang_makan = [0]
    maks_bersamaan = [0]
    lock_stat = threading.Lock()

    def filsuf(i):
        kiri = forks[i]
        kanan = forks[(i + 1) % N]
        for _ in range(putaran):
            with kursi_meja:
                kiri.acquire()
                kanan.acquire()
                with lock_stat:
                    sedang_makan[0] += 1
                    maks_bersamaan[0] = max(maks_bersamaan[0], sedang_makan[0])
                selesai_makan[i] += 1
                time.sleep(waktu_makan)
                with lock_stat:
                    sedang_makan[0] -= 1
                kanan.release()
                kiri.release()
            time.sleep(waktu_pikir)

    threads = [threading.Thread(target=filsuf, args=(i,)) for i in range(N)]
    mulai = time.perf_counter()
    for t in threads:
        t.start()
    for t in threads:
        t.join()
    durasi = time.perf_counter() - mulai
    assert sum(selesai_makan) == N * putaran, "Ada filsuf yang tidak selesai (deadlock?)"
    return sum(selesai_makan), durasi, maks_bersamaan[0]

# Sanity check: N = 5 (kasus di materi) harus selesai tanpa deadlock
total, durasi, mb = jalankan_arbitrator(5)
print(f"N=5 -> total makan {total}, durasi {durasi:.2f} dtk, maks makan bersamaan = {mb}")


N=5 -> total makan 100, durasi 1.02 dtk, maks makan bersamaan = 2


In [2]:
ULANGAN = 3          # tiap N diulang 3x lalu diambil rata-rata agar hasil lebih stabil
PUTARAN = 20
WAKTU_MAKAN = 0.02
hasil_n = {}

print(f"{'N':>3} | {'total makan':>11} | {'durasi (dtk)':>12} | {'throughput (makan/dtk)':>22} | {'maks makan bersamaan':>20} | {'batas ideal (floor(N/2)/t_makan)':>32}")
print("-" * 118)
for N in (3, 7, 10):
    throughputs, durasis, maks_list = [], [], []
    for _ in range(ULANGAN):
        total, durasi, mb = jalankan_arbitrator(N, putaran=PUTARAN, waktu_makan=WAKTU_MAKAN)
        throughputs.append(total / durasi)
        durasis.append(durasi)
        maks_list.append(mb)
    tp = statistics.mean(throughputs)
    ideal = (N // 2) / WAKTU_MAKAN
    hasil_n[N] = tp
    print(f"{N:>3} | {N*PUTARAN:>11} | {statistics.mean(durasis):>12.2f} | {tp:>22.1f} | {max(maks_list):>20} | {ideal:>32.1f}")

print()
dasar = hasil_n[3]
for N in (3, 7, 10):
    print(f"N={N:>2}: throughput {hasil_n[N]:6.1f} makan/dtk  ->  {hasil_n[N]/dasar:4.2f}x throughput N=3   (rasio N terhadap N=3 = {N/3:4.2f}x, rasio floor(N/2) = {(N//2)/1:4.2f}x)")


  N | total makan | durasi (dtk) | throughput (makan/dtk) | maks makan bersamaan | batas ideal (floor(N/2)/t_makan)
----------------------------------------------------------------------------------------------------------------------


  3 |          60 |         1.22 |                   49.3 |                    1 |                             50.0


  7 |         140 |         0.96 |                  146.2 |                    3 |                            150.0


 10 |         200 |         0.82 |                  245.1 |                    5 |                            250.0

N= 3: throughput   49.3 makan/dtk  ->  1.00x throughput N=3   (rasio N terhadap N=3 = 1.00x, rasio floor(N/2) = 1.00x)
N= 7: throughput  146.2 makan/dtk  ->  2.97x throughput N=3   (rasio N terhadap N=3 = 2.33x, rasio floor(N/2) = 3.00x)
N=10: throughput  245.1 makan/dtk  ->  4.97x throughput N=3   (rasio N terhadap N=3 = 3.33x, rasio floor(N/2) = 5.00x)


### Analisis Soal 1

**Jawaban: tidak, throughput tidak naik linear (proporsional) terhadap N.** Throughput mengikuti **⌊N/2⌋**, bukan N.

1. **Setiap filsuf butuh 2 garpu, padahal garpu hanya ada N.** Maka paling banyak **⌊N/2⌋ filsuf** bisa makan pada saat yang sama. Kolom *maks makan bersamaan* membuktikannya: N=3 → 1, N=7 → 3, N=10 → 5.
2. **Throughput mendekati batas ideal ⌊N/2⌋ / t_makan** (±50, ±150, ±250 makan/detik). Selisih kecilnya berasal dari waktu berpikir dan overhead penjadwalan thread.
3. **Perbandingan rasio:** dari N=3 ke N=7, jumlah filsuf naik 2,33×, tetapi throughput naik sekitar 3×. Dari N=3 ke N=10, filsuf naik 3,33×, throughput naik sekitar 5×. Jadi throughput **tidak proporsional dengan N**; ia proporsional dengan ⌊N/2⌋. Lompatannya "lebih besar dari N" karena N=3 kasus terburuk: hanya 1 filsuf yang bisa makan, sehingga praktis berjalan serial (efisiensi 1/3 dari jumlah filsuf, sedangkan N=10 efisiensi 1/2).
4. **Bukan `Semaphore(N-1)` yang menjadi batas throughput, melainkan jumlah garpu.** Semaphore hanya mencegah deadlock. Menambah filsuf tidak otomatis menambah kecepatan per filsuf; untuk N besar, throughput total tumbuh sekitar N/(2·t_makan), yaitu hanya separuh dari "kecepatan teoretis" jika setiap filsuf bisa makan sendiri-sendiri. Pada N ganjil, satu garpu selalu menganggur.
5. **Konsekuensi (sesuai catatan materi):** arbitrator adalah titik sinkronisasi global yang sedikit membatasi konkurensi, tetapi karena batas fisik garpu sudah lebih ketat (⌊N/2⌋ < N-1), pengaruh tambahan semaphore ini kecil.


---
## Soal 2 — Readers-Writers dengan Prioritas Writer

**Instruksi:** modifikasi solusi fair (Bagian 2) menjadi solusi yang **secara eksplisit memprioritaskan writer**: begitu ada writer menunggu, **tidak ada reader baru** yang boleh mulai membaca sampai writer selesai (reader yang sudah membaca boleh menyelesaikan bacaannya). Lalu uji apakah reader baru berisiko starvation.

**Rancangan (Second Readers-Writers Problem):**

| Primitif | Fungsi |
|---|---|
| `read_try` (Lock) | "Pintu masuk" reader. **Writer pertama yang menunggu menutupnya**, writer terakhir yang selesai membukanya. Reader baru harus lewat pintu ini, jadi tertahan selama ada writer menunggu/menulis. |
| `resource` (Lock) | Akses eksklusif ke data. Dipegang reader pertama (mewakili semua reader) atau writer yang sedang menulis. |
| `read_count` + `mutex_r` | Sama seperti solusi reader-preference: reader pertama mengunci `resource`, reader terakhir melepasnya. |
| `write_count` + `mutex_w` | Menghitung writer yang **sedang menulis atau menunggu**. Saat berubah 0→1 pintu reader ditutup; saat 1→0 pintu dibuka. |

Perbedaan utama dari solusi fair di materi: pada solusi fair, `antrian_masuk` bersifat adil ke semua pihak (FIFO), sedangkan di sini **writer selalu didahulukan** atas reader yang belum masuk.

### Uji 1 — Apakah prioritas writer bekerja?
Skenario: Reader-A dan Reader-B sudah membaca → Writer-W datang dan menunggu → Reader-C dan Reader-D datang **setelah** writer menunggu.
Hasil yang diharapkan: A dan B boleh menyelesaikan bacaan, lalu W menulis, **baru** C dan D membaca (dan melihat nilai baru).


In [3]:
import threading
import time

class ReadersWritersPrioritasWriter:
    """
    Writer-preference (Second Readers-Writers Problem).
    - read_try  : "pintu masuk" reader. Writer pertama yang menunggu MENUTUP pintu ini,
                  writer terakhir yang selesai MEMBUKANYA kembali.
    - resource  : akses eksklusif ke data (dipegang reader pertama / writer yang sedang menulis).
    """
    def __init__(self):
        self.read_try = threading.Lock()
        self.resource = threading.Lock()
        self.mutex_r = threading.Lock()
        self.mutex_w = threading.Lock()
        self.read_count = 0
        self.write_count = 0   # jumlah writer yang sedang menulis ATAU menunggu

    def mulai_baca(self):
        with self.read_try:                 # reader baru harus lewat pintu; tertutup jika ada writer menunggu
            with self.mutex_r:
                self.read_count += 1
                if self.read_count == 1:
                    self.resource.acquire() # reader pertama mengunci data dari writer

    def selesai_baca(self):
        with self.mutex_r:
            self.read_count -= 1
            if self.read_count == 0:
                self.resource.release()     # reader terakhir membuka data untuk writer

    def mulai_tulis(self):
        with self.mutex_w:
            self.write_count += 1
            if self.write_count == 1:
                self.read_try.acquire()     # writer pertama menutup pintu reader BARU
        self.resource.acquire()             # tunggu reader yang sudah di dalam selesai, lalu eksklusif

    def selesai_tulis(self):
        self.resource.release()
        with self.mutex_w:
            self.write_count -= 1
            if self.write_count == 0:
                self.read_try.release()     # tidak ada writer lagi -> pintu reader dibuka

# ---------- Uji 1: urutan akses ----------
rw = ReadersWritersPrioritasWriter()
data = {"nilai": 0}
log, log_lock = [], threading.Lock()
T0 = time.perf_counter()

def catat(msg):
    with log_lock:
        log.append(f"[{time.perf_counter()-T0:5.2f}s] {msg}")

def pembaca(nama, lama):
    catat(f"{nama} ingin membaca")
    rw.mulai_baca()
    catat(f"{nama} MULAI membaca (nilai={data['nilai']})")
    time.sleep(lama)
    catat(f"{nama} selesai membaca")
    rw.selesai_baca()

def penulis(nama, lama):
    catat(f"{nama} ingin menulis")
    rw.mulai_tulis()
    data["nilai"] += 1
    catat(f"{nama} MULAI menulis (nilai baru={data['nilai']})")
    time.sleep(lama)
    catat(f"{nama} selesai menulis")
    rw.selesai_tulis()

jadwal = [
    (0.00, pembaca, ("Reader-A", 0.30)),   # sudah membaca sebelum writer datang
    (0.05, pembaca, ("Reader-B", 0.30)),   # sudah membaca sebelum writer datang
    (0.10, penulis, ("Writer-W", 0.10)),   # writer datang & harus menunggu A, B
    (0.15, pembaca, ("Reader-C", 0.05)),   # reader BARU datang SETELAH writer menunggu
    (0.20, pembaca, ("Reader-D", 0.05)),   # reader BARU lainnya
]
threads = []
for delay, fn, args in jadwal:
    time.sleep(max(0, delay - (time.perf_counter() - T0)))
    t = threading.Thread(target=fn, args=args)
    t.start()
    threads.append(t)
for t in threads:
    t.join()

print("\n".join(log))
print(f"\nNilai akhir = {data['nilai']} (harus 1)")


[ 0.00s] Reader-A ingin membaca
[ 0.00s] Reader-A MULAI membaca (nilai=0)
[ 0.05s] Reader-B ingin membaca
[ 0.05s] Reader-B MULAI membaca (nilai=0)
[ 0.10s] Writer-W ingin menulis
[ 0.15s] Reader-C ingin membaca
[ 0.20s] Reader-D ingin membaca
[ 0.30s] Reader-A selesai membaca
[ 0.35s] Reader-B selesai membaca
[ 0.35s] Writer-W MULAI menulis (nilai baru=1)
[ 0.45s] Writer-W selesai menulis
[ 0.45s] Reader-C MULAI membaca (nilai=1)
[ 0.45s] Reader-D MULAI membaca (nilai=1)
[ 0.50s] Reader-D selesai membaca
[ 0.50s] Reader-C selesai membaca

Nilai akhir = 1 (harus 1)


**Analisis Uji 1:**
- Reader-A dan Reader-B yang **sudah membaca** sebelum writer datang tidak diganggu dan menyelesaikan bacaannya (sesuai instruksi).
- Reader-C dan Reader-D, yang datang **setelah** Writer-W menunggu, tidak boleh mulai membaca. Mereka baru mulai setelah W selesai menulis. Pada solusi *reader preference* di materi, C dan D akan "menyerobot" masuk dan W makin lama menunggu.
- Reader-C dan Reader-D membaca **nilai=1** (hasil tulisan W), bukan nilai lama. Ini ciri khas prioritas writer: data yang dibaca reader lebih segar.
- Nilai akhir = 1, tidak ada *lost update* maupun deadlock.

### Uji 2 — Apakah reader baru berisiko starvation jika writer datang terus-menerus?
Skenario: 3 writer bekerja berulang-ulang dengan jeda 1 ms, lalu satu reader mencoba masuk. Reader diberi waktu 2 detik; setelah itu writer dihentikan.


In [4]:
# ---------- Uji 2: apakah READER bisa starvation jika writer datang terus-menerus? ----------
rw2 = ReadersWritersPrioritasWriter()
berhenti = threading.Event()
jumlah_tulis = [0]
reader_masuk = threading.Event()
lock_hitung = threading.Lock()

def writer_terus_menerus():
    while not berhenti.is_set():
        rw2.mulai_tulis()
        time.sleep(0.01)
        with lock_hitung:
            jumlah_tulis[0] += 1
        rw2.selesai_tulis()
        time.sleep(0.001)   # jeda sangat singkat sebelum writer berikutnya datang

waktu_tunggu_reader = [None]
def satu_pembaca():
    t0 = time.perf_counter()
    rw2.mulai_baca()
    waktu_tunggu_reader[0] = time.perf_counter() - t0
    reader_masuk.set()
    rw2.selesai_baca()

writers = [threading.Thread(target=writer_terus_menerus) for _ in range(3)]
for t in writers:
    t.start()
time.sleep(0.1)                     # biarkan writer ramai lebih dulu
reader = threading.Thread(target=satu_pembaca)
reader.start()

reader.join(timeout=2)              # beri reader waktu 2 detik
if not reader_masuk.is_set():
    print(f"Reader BELUM mendapat giliran setelah 2 detik -- STARVATION READER TERBUKTI!")
    print(f"Jumlah penulisan yang terjadi selama reader menunggu: {jumlah_tulis[0]}")
else:
    print("Reader sempat masuk (tidak starvation pada percobaan ini)")

berhenti.set()                      # hentikan writer; reader seharusnya akhirnya masuk
for t in writers:
    t.join()
reader.join()
print(f"Setelah writer berhenti, reader akhirnya masuk. Total waktu tunggu reader = {waktu_tunggu_reader[0]:.2f} detik")
print(f"Total penulisan selama itu: {jumlah_tulis[0]}")


Reader BELUM mendapat giliran setelah 2 detik -- STARVATION READER TERBUKTI!
Jumlah penulisan yang terjadi selama reader menunggu: 207
Setelah writer berhenti, reader akhirnya masuk. Total waktu tunggu reader = 2.02 detik
Total penulisan selama itu: 210


**Analisis Uji 2 — Ya, reader berisiko starvation:**
- Writer yang datang terus-menerus membuat `write_count` hampir tidak pernah mencapai 0, sehingga pintu `read_try` **tetap tertutup**. Reader menunggu sampai writer berhenti, padahal ratusan penulisan sudah terjadi.
- Ini adalah **cermin terbalik** dari Bagian 2 pada materi: *reader preference* → **writer starvation**; *writer preference* → **reader starvation**.
- Dalam bahasa materi Pertemuan 5, solusi ini melanggar syarat **bounded waiting**: reader tidak punya batas atas waktu tunggu.
- **Kesimpulan trade-off:** prioritas writer cocok bila data harus selalu segar dan penulisan jarang (mis. konfigurasi atau cache yang diperbarui sesekali). Bila penulisan sangat sering, perlu solusi yang benar-benar adil (FIFO ketat antara reader dan writer, Third Readers-Writers Problem) agar tidak ada pihak yang kelaparan.


---
## Soal 3 — Sleeping Barber dengan 2 Tukang Cukur

**Instruksi:** ubah Bagian 4 menjadi **2 tukang cukur** yang berbagi satu ruang tunggu; tentukan bagaimana memilih tukang cukur yang melayani pelanggan berikutnya (hint: `Semaphore` tambahan atau `Queue`).

**Rancangan:**
- **Ruang tunggu bersama = satu `queue.Queue`.** Pelanggan yang dapat kursi memasukkan dirinya ke antrean; tukang cukur memanggil `antrean.get()`.
- **Menentukan siapa melayani siapa:** tidak perlu penjadwal khusus. Kedua tukang memanggil `get()` pada antrean yang sama, sehingga **tukang yang bebas lebih dulu otomatis mendapat pelanggan terdepan (FIFO)**. `Queue.get()` bersifat thread-safe dan memblok jika kosong, jadi "tidur saat tidak ada pelanggan" terpenuhi tanpa busy-waiting.
- **Kapasitas terbatas (3 kursi tunggu):** penghitung `menunggu` dilindungi `mutex`. Pelanggan yang datang saat penghitung sudah 3 langsung **pergi** (rejection), seperti pada materi. Kursi dihitung kosong kembali ketika tukang cukur mengambil pelanggan dari antrean.
- **Sinyal selesai:** tiap pelanggan membawa `threading.Event` sendiri, sehingga ia tahu persis kapan *dirinya* selesai dicukur oleh tukang mana pun. (Pada versi 1 tukang di materi dipakai tiga semaphore bersama; dengan 2 tukang semaphore bersama tidak bisa membedakan pelanggan mana yang sudah selesai.)
- **Penghentian:** setelah semua pelanggan ditangani, dikirim satu sinyal `None` per tukang cukur agar thread berhenti dengan bersih.
- Kedatangan pelanggan dipercepat (jeda 0,01–0,05 detik) agar kasus "ruang tunggu penuh" terlihat walau ada 2 tukang.


In [5]:
import threading
import queue
import time
import random

def simulasi_barbershop(jumlah_tukang, jumlah_pelanggan=14, kursi_tunggu=3, waktu_cukur=0.15, jeda=(0.01, 0.05), seed=1, verbose=True):
    antrean = queue.Queue()            # ruang tunggu BERSAMA; get() yang memblok = tukang cukur "tidur"
    mutex = threading.Lock()           # melindungi penghitung kursi tunggu
    menunggu = [0]
    log_lock = threading.Lock()
    dilayani_oleh = {k: [] for k in range(jumlah_tukang)}
    pergi = []

    def catat(msg):
        if verbose:
            with log_lock:
                print(msg)

    def tukang_cukur(id_t):
        while True:
            if antrean.empty():
                catat(f"[Tukang-{id_t}] tidur, menunggu pelanggan...")
            item = antrean.get()       # siapa pun tukang yang bebas duluan mendapat pelanggan berikutnya (FIFO)
            if item is None:           # sinyal berhenti
                break
            id_p, selesai = item
            with mutex:
                menunggu[0] -= 1       # kursi tunggu kosong lagi
            catat(f"[Tukang-{id_t}] mencukur Pelanggan-{id_p}")
            time.sleep(waktu_cukur)
            dilayani_oleh[id_t].append(id_p)
            selesai.set()

    def pelanggan(id_p):
        with mutex:
            if menunggu[0] < kursi_tunggu:
                menunggu[0] += 1
                catat(f"[Pelanggan-{id_p}] duduk di ruang tunggu ({menunggu[0]}/{kursi_tunggu})")
                duduk = True
            else:
                catat(f"[Pelanggan-{id_p}] ruang tunggu PENUH, pergi!")
                pergi.append(id_p)
                duduk = False
        if duduk:
            selesai = threading.Event()
            antrean.put((id_p, selesai))
            selesai.wait()

    tukang = [threading.Thread(target=tukang_cukur, args=(k,), daemon=True) for k in range(jumlah_tukang)]
    for t in tukang:
        t.start()

    random.seed(seed)
    plg = []
    for i in range(jumlah_pelanggan):
        time.sleep(random.uniform(*jeda))
        t = threading.Thread(target=pelanggan, args=(i,))
        t.start()
        plg.append(t)
    for t in plg:
        t.join()
    for _ in tukang:                   # satu sinyal berhenti per tukang cukur
        antrean.put(None)
    for t in tukang:
        t.join()
    return dilayani_oleh, pergi

print("===== SIMULASI: 2 TUKANG CUKUR, 1 RUANG TUNGGU (3 KURSI), 14 PELANGGAN =====")
dilayani, pergi = simulasi_barbershop(jumlah_tukang=2)
print("\n----- RINGKASAN -----")
for k, daftar in dilayani.items():
    print(f"Tukang-{k} melayani {len(daftar)} pelanggan: {daftar}")
total_dilayani = sum(len(v) for v in dilayani.values())
print(f"Pelanggan pergi (ruang tunggu penuh): {len(pergi)} -> {pergi}")
print(f"Dilayani + pergi = {total_dilayani} + {len(pergi)} = {total_dilayani + len(pergi)} (harus 14)")


===== SIMULASI: 2 TUKANG CUKUR, 1 RUANG TUNGGU (3 KURSI), 14 PELANGGAN =====
[Tukang-0] tidur, menunggu pelanggan...
[Tukang-1] tidur, menunggu pelanggan...
[Pelanggan-0] duduk di ruang tunggu (1/3)
[Tukang-0] mencukur Pelanggan-0
[Pelanggan-1] duduk di ruang tunggu (1/3)
[Tukang-1] mencukur Pelanggan-1
[Pelanggan-2] duduk di ruang tunggu (1/3)
[Pelanggan-3] duduk di ruang tunggu (2/3)
[Pelanggan-4] duduk di ruang tunggu (3/3)
[Tukang-0] mencukur Pelanggan-2


[Pelanggan-5] duduk di ruang tunggu (3/3)


[Tukang-1] mencukur Pelanggan-3
[Pelanggan-6] duduk di ruang tunggu (3/3)
[Pelanggan-7] ruang tunggu PENUH, pergi!
[Pelanggan-8] ruang tunggu PENUH, pergi!
[Pelanggan-9] ruang tunggu PENUH, pergi!
[Tukang-0] mencukur Pelanggan-4
[Pelanggan-10] duduk di ruang tunggu (3/3)
[Pelanggan-11] ruang tunggu PENUH, pergi!
[Tukang-1] mencukur Pelanggan-5


[Pelanggan-12] duduk di ruang tunggu (3/3)
[Pelanggan-13] ruang tunggu PENUH, pergi!


[Tukang-0] mencukur Pelanggan-6
[Tukang-1] mencukur Pelanggan-10


[Tukang-0] mencukur Pelanggan-12
[Tukang-1] tidur, menunggu pelanggan...


[Tukang-0] tidur, menunggu pelanggan...

----- RINGKASAN -----
Tukang-0 melayani 5 pelanggan: [0, 2, 4, 6, 12]
Tukang-1 melayani 4 pelanggan: [1, 3, 5, 10]
Pelanggan pergi (ruang tunggu penuh): 5 -> [7, 8, 9, 11, 13]
Dilayani + pergi = 9 + 5 = 14 (harus 14)


In [6]:
# Perbandingan: 1 tukang cukur vs 2 tukang cukur (skenario pelanggan yang sama, tanpa log detail)
print(f"{'Tukang':>6} | {'Dilayani':>8} | {'Pergi':>5}")
print("-" * 28)
for jt in (1, 2):
    dl, pg = simulasi_barbershop(jumlah_tukang=jt, verbose=False)
    print(f"{jt:>6} | {sum(len(v) for v in dl.values()):>8} | {len(pg):>5}")


Tukang | Dilayani | Pergi
----------------------------


     1 |        6 |     8


     2 |        9 |     5


### Analisis Soal 3
- **Pembagian kerja:** kedua tukang cukur melayani pelanggan dalam jumlah hampir seimbang, dan pelanggan dilayani **berurutan sesuai antrean** (FIFO). Tukang mana yang melayani tidak ditentukan sebelumnya, melainkan oleh siapa yang selesai lebih dulu: itulah mekanisme "giliran antar-tukang" yang diminta soal.
- **Rejection tetap terjadi:** ketika pelanggan datang lebih cepat dari kapasitas layanan, ruang tunggu penuh dan pelanggan pergi. Ini analog dengan server yang menolak koneksi saat *connection pool* penuh.
- **Konservasi:** dilayani + pergi = jumlah total pelanggan, jadi tidak ada pelanggan yang hilang atau terjebak menunggu selamanya.
- **Perbandingan 1 vs 2 tukang:** dengan skenario pelanggan yang sama, 2 tukang melayani lebih banyak pelanggan dan menolak lebih sedikit dibanding 1 tukang. Menambah "worker" menaikkan kapasitas, tetapi ruang tunggu (buffer) yang terbatas tetap menimbulkan penolakan jika beban melebihi kapasitas.


---
## Soal 4 — Refleksi Singkat (maks. 150 kata)

**Pertanyaan:** dari kelima classical problem, mana yang **paling mirip** dengan masalah nyata dalam pengembangan software? Berikan contoh konkret.


> Menurut saya yang paling mirip adalah Producer-Consumer, dengan unsur Sleeping Barber. Contohnya web server: thread penerima request (producer) memasukkan tugas ke antrean, lalu worker thread (consumer) mengambil dan memprosesnya. Worker yang menganggur "tidur" pada get() yang memblok tanpa memakan CPU. Karena antreannya berkapasitas terbatas, request yang datang saat antrean penuh ditolak (misalnya HTTP 503), persis seperti pelanggan yang pergi pada Sleeping Barber. Pola yang sama muncul pada sistem embedded, yaitu interrupt handler sensor sebagai producer dan thread pengolah data sebagai consumer. Pelajarannya: ukuran buffer dan kebijakan saat penuh (menunggu atau menolak) harus dirancang sengaja, dan jenis Queue (thread, proses, asyncio) dipilih sesuai beban kerjanya.

*(Jumlah kata: 104)*

---
## Kesimpulan Latihan Mandiri

| Soal | Temuan utama |
|---|---|
| 1. Dining Philosophers N=3,7,10 | Throughput mengikuti ⌊N/2⌋ (batas garpu), tidak proporsional dengan N. |
| 2. Readers-Writers prioritas writer | Reader baru tertahan saat writer menunggu (data lebih segar), tetapi reader bisa **starvation** jika writer datang terus-menerus. |
| 3. Sleeping Barber 2 tukang | Satu `Queue` bersama sudah cukup untuk membagi pelanggan antar-tukang secara FIFO; kapasitas terbatas tetap menimbulkan rejection. |
| 4. Refleksi | Producer-Consumer + Sleeping Barber paling mirip dengan web server/task queue. |
